In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Chargement avec parsing des dates
file_path = "../data/raw/velib-2026-10-08.csv.gz"
df = pd.read_csv(file_path, parse_dates=["collected_at_utc", "source_updated_at"])
#print(df.columns.tolist())

# Conversion directe UTC -> Heure de Paris (UTC+2)
df['collected_at_paris'] = df['collected_at_utc'].dt.tz_convert('Europe/Paris')
df['source_updated_at_paris'] = df['source_updated_at'].dt.tz_convert('Europe/Paris')

# Tri chronologique par station pour faciliter l'analyse
df = df.sort_values(by=['station_code', 'collected_at_paris']).reset_index(drop=True)

display(df.head())
df.info()

,collected_at_utc,source_updated_at,station_code,name,commune,insee_code,latitude,longitude,capacity,bikes_available,mechanical_bikes,electric_bikes,docks_available,is_installed,is_renting,is_returning,collected_at_paris,source_updated_at_paris
0,2026-08-25 11:57:26+00:00,2026-08-25 11:31:19+00:00,2102,Saint-Marc - Feydeau,Paris,75056,48.870327,2.341872,37.0,35,20,15,2,1,1,1,2026-08-25 13:57:26+02:00,2026-08-25 13:31:19+02:00
1,2026-08-25 12:00:02+00:00,2026-08-25 11:31:19+00:00,2102,Saint-Marc - Feydeau,Paris,75056,48.870327,2.341872,37.0,34,20,14,3,1,1,1,2026-08-25 14:00:02+02:00,2026-08-25 13:31:19+02:00
2,2026-08-25 12:15:01+00:00,2026-08-25 11:31:19+00:00,2102,Saint-Marc - Feydeau,Paris,75056,48.870327,2.341872,37.0,33,19,14,4,1,1,1,2026-08-25 14:15:01+02:00,2026-08-25 13:31:19+02:00
3,2026-08-25 12:30:02+00:00,2026-08-25 11:31:19+00:00,2102,Saint-Marc - Feydeau,Paris,75056,48.870327,2.341872,37.0,34,19,15,3,1,1,1,2026-08-25 14:30:02+02:00,2026-08-25 13:31:19+02:00
4,2026-08-25 12:45:02+00:00,2026-08-25 11:31:19+00:00,2102,Saint-Marc - Feydeau,Paris,75056,48.870327,2.341872,37.0,33,19,14,4,1,1,1,2026-08-25 14:45:02+02:00,2026-08-25 13:31:19+02:00


<class 'pandas.DataFrame'>
RangeIndex: 329680 entries, 0 to 329679
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype                       
---  ------                   --------------   -----                       
 0   collected_at_utc         329680 non-null  datetime64[us, UTC]         
 1   source_updated_at        329680 non-null  datetime64[us, UTC]         
 2   station_code             329680 non-null  int64                       
 3   name                     329680 non-null  str                         
 4   commune                  329680 non-null  str                         
 5   insee_code               329680 non-null  int64                       
 6   latitude                 329680 non-null  float64                     
 7   longitude                329680 non-null  float64                     
 8   capacity                 327760 non-null  float64                     
 9   bikes_available          329680 non-null  int64            

## Projection à t + 30 min

In [34]:
df[['bikes_futur', 'time_futur']] = (
    df.groupby('station_code')[['bikes_available', 'collected_at_utc']]
    .shift(-2)
)

## Ecart temporel (vérifier si les écart sont d'éxactement 30min)

In [35]:
df['delta_minutes'] = (df['time_futur'] - df['collected_at_utc']).dt.total_seconds() / 60

# Inspection des statistiques et des valeurs les plus fréquentes
print(df['delta_minutes'].describe())
print("\nTop 10 des écarts observés (en minutes) :")
print(df['delta_minutes'].value_counts(dropna=False).head(10))

count    329520.000000
mean         31.002084
std          36.720510
min          17.583333
25%          29.983333
50%          30.000000
75%          30.016667
max        1695.016667
Name: delta_minutes, dtype: float64

Top 10 des écarts observés (en minutes) :
delta_minutes
30.000000      157730
29.983333       83630
30.016667       83350
45.000000        2300
44.983333         760
45.016667         720
30.033333         240
29.966667         160
1695.016667       160
NaN               160
Name: count, dtype: int64


* **`NaN = 160` :** Correspond exactement aux 2 dernières lignes sans futur des 80 stations (`80 × 2`), confirmant l'absence de mélange entre stations.
* **Latence de l'API (`29,98` à `30,03` min) :** Décalage normal de ±1 à 2 s lors de la collecte. Une tolérance (ex. `[29, 31]` min) est indispensable pour ne pas écarter plus de 167 000 observations valides.
* **Trous de collecte à filtrer :** Écarts anormaux (`17,58` min, `45` min pour un relevé manquant, `1695` min soit ~28 h de coupure générale) à supprimer pour garantir un horizon strict de 30 minutes.

### Filtrage des intervalles valides (tolérance [29, 31] min) et copie propre

In [36]:
n_avant = len(df)
df = df[df['delta_minutes'].between(29, 31)].copy()
print(f"Lignes conservées : {len(df)} / {n_avant} ({n_avant - len(df)} lignes écartées)")

Lignes conservées : 325190 / 329680 (4490 lignes écartées)


### Création de la variable cible binaire (1 si < 2 vélos à t+30 min, 0 sinon)

In [37]:
df['cible'] = (df['bikes_futur'] < 2).astype(int)
df = df.drop(columns=['bikes_futur', 'time_futur', 'delta_minutes'])

### Mesure du déséquilibre des classes 

In [38]:
print("\nRépartition de la cible (%) :")
print((df['cible'].value_counts(normalize=True) * 100).round(2))


Répartition de la cible (%) :
cible
0    88.82
1    11.18
Name: proportion, dtype: float64


## Création de la variable pred_persitance 

In [39]:
df['pred_persistance'] = (df['bikes_available'] < 2).astype(int)

### Matrice de confusion 

In [40]:
print("Matrice de confusion :")
display(pd.crosstab(df['cible'], df['pred_persistance'], rownames=['Réalité (t+30)'], colnames=['Persistance (t)']))
print("\nPerformances de la baseline de persistance :")
print(classification_report(df['cible'], df['pred_persistance'], digits=4))

Matrice de confusion :


Persistance (t),0,1
Réalité (t+30),,
0,279076,9748
1,9722,26644



Performances de la baseline de persistance :
              precision    recall  f1-score   support

           0     0.9663    0.9662    0.9663    288824
           1     0.7321    0.7327    0.7324     36366

    accuracy                         0.9401    325190
   macro avg     0.8492    0.8495    0.8493    325190
weighted avg     0.9401    0.9401    0.9401    325190



* **Score de référence** : La référence à battre est le **F1-score de 0,7324** sur la classe `1`.
* **Vrais Négatifs (279 076 — ~86 %) :** Stations restant approvisionnées entre $t$ et $t+30$, ce qui gonfle artificiellement l'accuracy sans mesurer la détection des pénuries.
* **Faux Négatifs (9 722 — Rappel 73,27 %) :** Stations vidées en moins de 30 minutes (transitions critiques invisibles pour la persistance, cible prioritaire des futurs modèles ML).
* **Faux Positifs (9 748 — Précision 73,21 %) :** Stations en pénurie à $t$ réapprovisionnées ou recevant des retours d'usagers avant $t+30$.

## Découpage Train / Test / Validation

### Contraintes de découpage 

* **Alignement sur des journées complètes :** Arrondir les coupures à minuit (`00:00:00`) pour ne pas couper les cycles journaliers (pointes matin/soir) et conserver un mélange représentatif de jours ouvrés et de week-ends dans chaque bloc.
* **Embargo de 30 minutes :** Laisser un écart d'au moins 30 minutes entre la fin d'un sous-ensemble et le début du suivant afin qu'aucune cible (calculée à $t + 30\text{ min}$) ne morde sur la période d'évaluation.
* **Stabilité du taux de pénurie :** Vérifier que la proportion de `cible == 1` reste proche des **11,18 %** globaux dans chacun des trois blocs (`Train`, `Validation`, `Test`).

### Etendue temporelle du jeu de données

In [41]:
t_min = df['collected_at_paris'].min()
t_max = df['collected_at_paris'].max()
duree = t_max - t_min

print(f"Début de collecte : {t_min}")
print(f"Fin de collecte   : {t_max}")
print(f"Durée totale      : {duree}")

# Calcul des seuils temporels théoriques (70% / 85%)
date_val = df['collected_at_paris'].quantile(0.70)
date_test = df['collected_at_paris'].quantile(0.85)

print(f"\nSeuil suggéré Train -> Val  : {date_val}")
print(f"Seuil suggéré Val   -> Test : {date_test}")

Début de collecte : 2026-08-25 14:00:02+02:00
Fin de collecte   : 2026-10-08 21:45:02+02:00
Durée totale      : 44 days 07:45:00

Seuil suggéré Train -> Val  : 2026-09-26 00:00:02+02:00
Seuil suggéré Val   -> Test : 2026-10-02 10:45:02+02:00


* **Train (~31,5 jours — ~71 %) :** Du 25 août au 25 septembre (coupure au `2026-09-26 00:00:00`, soit 4 week-ends complets et 24 jours ouvrés).
* **Validation (6 jours — ~13,5 %) :** Du 26 septembre au 1er octobre (coupure au `2026-10-02 00:00:00`, soit 1 week-end complet et 4 jours ouvrés).
* **Test (~7 jours — ~15,5 %) :** Du 2 au 8 octobre (1 week-end complet et 5 jours ouvrés).
* **Équilibre calendaire :** Garantit un week-end complet en Validation et en Test afin d'évaluer le modèle sur les dynamiques de week-end, très différentes des heures de pointe en semaine.

In [42]:
# Définition des frontières temporelles (fuseau Europe/Paris)
tz = df['collected_at_paris'].dt.tz
cutoff_val = pd.Timestamp('2026-09-26 00:00:00', tz=tz)
cutoff_test = pd.Timestamp('2026-10-02 00:00:00', tz=tz)
embargo = pd.Timedelta(minutes=30)

# Découpage chronologique strict avec embargo de 30 min
train_df = df[df['collected_at_paris'] < (cutoff_val - embargo)].copy()
val_df = df[
    (df['collected_at_paris'] >= cutoff_val) & 
    (df['collected_at_paris'] < (cutoff_test - embargo))
].copy()
test_df = df[df['collected_at_paris'] >= cutoff_test].copy()

# Vérification des volumes et de la stabilité de la cible
for name, subset in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    pct_rows = len(subset) / len(df) * 100
    pct_pos = subset['cible'].mean() * 100
    print(f"{name:<10} : {len(subset):>6} lignes ({pct_rows:5.2f}%) | Taux de pénurie (cible=1) : {pct_pos:.2f}%")

# Score de référence officiel de la persistance sur le jeu de TEST uniquement
print("\n--- Référence Persistance sur le jeu de TEST ---")
print(classification_report(test_df['cible'], test_df['pred_persistance'], digits=4))

Train      : 227420 lignes (69.93%) | Taux de pénurie (cible=1) : 10.57%
Validation :  45230 lignes (13.91%) | Taux de pénurie (cible=1) : 12.99%
Test       :  52220 lignes (16.06%) | Taux de pénurie (cible=1) : 12.32%

--- Référence Persistance sur le jeu de TEST ---
              precision    recall  f1-score   support

           0     0.9628    0.9626    0.9627     45789
           1     0.7339    0.7352    0.7346      6431

    accuracy                         0.9346     52220
   macro avg     0.8484    0.8489    0.8486     52220
weighted avg     0.9346    0.9346    0.9346     52220



* **Cible de performance** : Tout modèle de Machine Learning retenu devra dépasser F1 = 0,7346 sur le jeu de Test (en particulier en améliorant le Rappel de 73,52 %, là où la persistance rate les stations qui se vident brutalement).

In [43]:
print(df.columns.tolist())

['collected_at_utc', 'source_updated_at', 'station_code', 'name', 'commune', 'insee_code', 'latitude', 'longitude', 'capacity', 'bikes_available', 'mechanical_bikes', 'electric_bikes', 'docks_available', 'is_installed', 'is_renting', 'is_returning', 'collected_at_paris', 'source_updated_at_paris', 'cible', 'pred_persistance']
